# Informe Ejecutivo: Análisis de Ventas

**Resumen:** este cuaderno presenta los descubrimientos clave del análisis de ventas: producto(s) más vendido(s), ciudades con mayor consumo, relaciones entre cantidad/precio/importe, y recomendaciones accionables para la gerencia. El objetivo es ofrecer información clara y utilizable para la toma de decisiones.

*Fecha del análisis:* 2025-11-05



## Resumen ejecutivo (para gerencia)

- Producto más vendido (por cantidad): *salsa de tomate 500g* (top después de normalización).
- Producto que más aportó al ingreso (por importe): *desodorante aerosol*.
- Ciudad con mayor consumo total: *Rio Cuarto* (total ventas agregadas: 784,447 aprox.).
- Hallazgos estadísticos clave:
  - Existe una correlación fuerte y significativa entre `cantidad` e `importe` (r ≈ 0.60, p << 0.001).
  - Existe una correlación fuerte y significativa entre `precio_unitario` e `importe` (r ≈ 0.68, p << 0.001).
  - No hay correlación significativa entre `cantidad` y `precio_unitario` (r ≈ -0.07, p ≈ 0.18).

Estos hallazgos indican que tanto el número de unidades como el precio unitario contribuyen al ingreso; las estrategias pueden priorizar mix de producto (productos de alto importe) y promociones por ciudad.


In [ ]:
## Hallazgos clave — tablas y gráficos (código comentado)

# A continuación muestro el bloque de código que carga y presenta los outputs principales.
# He añadido comentarios que explican cada paso para que un trainee los entienda.

# Importar utilidades para visualización y manejo de archivos
from IPython.display import Image, display, Markdown
import pandas as pd
from pathlib import Path
out = Path('analysis_outputs')

# Mostrar métricas por ciudad: intentamos leer el CSV y enseñarlo
# Si no existe, imprimimos un mensaje claro para que el analista lo detecte
try:
    # city_metrics contiene columnas: ciudad, total_sales, n_sales, avg_ticket
    city_metrics = pd.read_csv(out / 'city_metrics_scipy.csv')
    display(Markdown('### Métricas por ciudad'))
    display(city_metrics.head(10))
except Exception as e:
    display(Markdown(f'**city_metrics_scipy.csv no disponible:** {e}'))

# Mostrar correlaciones (pares): CSV con r y p-values
try:
    corr_pairs = pd.read_csv(out / 'correlations_pairs_scipy.csv')
    display(Markdown('### Correlaciones (pares)'))
    display(corr_pairs)
except Exception as e:
    display(Markdown(f'**correlations_pairs_scipy.csv no disponible:** {e}'))

# Mostrar heatmap: si existe la imagen la mostramos; útil para presentación visual
p = out / 'correlation_heatmap_scipy.png'
if p.exists():
    display(Markdown('### Heatmap de correlación'))
    display(Image(filename=str(p)))
else:
    display(Markdown('**Heatmap no encontrado:** correlation_heatmap_scipy.png'))

# Mostrar top productos por cantidad y por ventas (archivos generados por el pipeline)
for fname, title in [('product_aggregation_qty.csv','Top productos por cantidad'),('product_aggregation_sales.csv','Top productos por ventas')]:
    p = out / fname
    display(Markdown(f'### {title}'))
    if p.exists():
        try:
            df = pd.read_csv(p)
            # Mostramos las primeras filas; para el trainee: revisar nombres y cantidades
            display(df.head(10))
        except Exception as e:
            display(Markdown(f'Error leyendo {fname}: {e}'))
    else:
        display(Markdown(f'**{fname} no encontrado**'))


## Interpretación detallada

- Producto más vendido (cantidad): el top por cantidad indica productos de alta rotación. Las decisiones: asegurar stock suficiente, revisar promociones que impulsan volumen.

- Producto por aporte a ingresos (importe): los artículos con alto importe deben ser analizados por margen; si el margen es favorable, priorizar su disponibilidad y merchandising.

- Ciudades: Rio Cuarto concentra el mayor volumen total. Para la gerencia esto significa priorizar logística y reposición allí; otras ciudades con alto `avg_ticket` pero menos transacciones (por ejemplo Villa María) son candidatas para aumentar frecuencia con campañas.



## Recomendaciones ejecutivas (priorizadas)

1. Operaciones / Inventario
   - Priorizar stock en Rio Cuarto para productos de alta rotación y alto aporte por importe.
   - Revisar proveedores y tiempos de reposición para evitar rupturas en los top SKUs.

2. Comercial / Marketing
   - Lanzar una campaña de fidelización en ciudades con alto `avg_ticket` pero baja frecuencia (ej. Villa María) para aumentar recurrencia.
   - Promociones cruzadas: combinar productos de alto importe con productos de alta rotación para aumentar ticket medio.

3. Finanzas / Producto
   - Calcular margen por producto (si hay datos de costo) para priorizar acciones sobre productos rentables.

4. Datos / Calidad
   - Limpiar `Detalle_ventas.csv` (eliminar columnas `Unnamed:*`) y validar unidades monetarias.
   - Mantener la normalización de nombres de producto y revisar manualmente el top 20 para corregir agrupamientos erróneos.



## Metodología (resumen técnico)

- Datos usados: `Detalle_ventas.csv`, `Ventas.csv`, `Clientes.csv`, `Productos.csv` (convertidos desde .xlsx y limpiados parcialmente).
- Pipeline: normalizamos nombres de producto, unimos `Detalle_ventas` con `Ventas` (por `id_venta`) y con `Clientes` (por `id_cliente`) para atribuir `ciudad` a cada línea de venta.
- Variables clave: `cantidad`, `precio_unitario`, `importe` (se generó `importe` calculado cuando faltaba sumando cantidad*precio_unitario).
- Análisis estadístico: correlación Pearson/Spearman entre variables numéricas; p-values calculados con `scipy`. Pruebas t para comparar ticket medio entre la ciudad top y el resto.



## Limitaciones y siguientes pasos

- Calidad de datos: limpiar `Unnamed:*`, revisar `precio_unitario` y `importe` por outliers.
- Reproducibilidad: ejecutar el notebook en Jupyter con `PYTHONUTF8=1` o usar una ruta sin caracteres especiales.
- Siguientes análisis sugeridos:
  - RFM y segmentación de clientes.
  - Cálculo de margen por producto (si hay costos) y priorización por rentabilidad.
  - Pruebas A/B en campañas para ciudades seleccionadas.



## Anexos: archivos generados y reproducibilidad

- Carpeta de outputs: `analysis_outputs/` — contiene CSVs y PNGs usados en este informe.
- Archivos clave:
  - `city_metrics_scipy.csv` — métricas por ciudad (total_sales, n_sales, avg_ticket)
  - `product_aggregation_qty.csv`, `product_aggregation_sales.csv` — agregados por producto
  - `correlations_pairs_scipy.csv`, `correlation_heatmap_scipy.png` — análisis estadístico
- Reproducir: ver `README.md` y `requirements.txt` en la raíz del proyecto.

---

*Fin del informe ejecutivo.*


# Comentarios para analistas (Explicación del código usado en este informe)

Este cuaderno está pensado para presentar resultados a la gerencia, pero también incluye una celda de código que carga y muestra tablas e imágenes desde `analysis_outputs/`.

1) Celda de carga y visualización (célula 3)
- Qué hace: usa `pandas.read_csv` para abrir archivos CSV generados por el pipeline (por ejemplo `city_metrics_scipy.csv`, `product_aggregation_sales.csv`) y `IPython.display.Image` para mostrar imágenes (heatmap). Muestra tablas con `display()` para que sean legibles en el notebook.
- Por qué es útil: separa la generación de resultados (que se hizo previamente con scripts) de la presentación; el notebook de gerencia no re-calcula todo, solo carga y presenta.

2) Buenas prácticas explicadas para un trainee:
- Mantén las rutas relativas (`analysis_outputs/`) para que el notebook funcione en cualquier máquina si se mantiene la estructura de carpetas.
- El notebook no debe reescribir ni modificar los CSVs de origen: su objetivo es presentar. Si necesitas reproducir un paso, haz una copia o crea un notebook separado de procesamiento.
- Para añadir un gráfico nuevo al informe: genera la imagen en `analysis_outputs/` con un script o celda separada y luego añade la referencia en la celda de presentación.

3) Checklist rápido antes de presentar a gerencia:
- Ejecutar la celda que carga las tablas y asegurarse de que las imágenes se muestran (heatmap y gráficos top city/product).
- Revisar textos del resumen ejecutivo (nombres de productos y cifras) y corregir manualmente si algún nombre aparece mal por la normalización.
- Exportar el notebook a PDF (desde Jupyter) o usar "Print to PDF" desde el navegador para obtener un documento listo para distribución.

Si quieres que deje comentarios inline (comentarios en el código) en la celda Python del cuaderno para explicar cada bloque de 3–4 líneas, lo hago ahora y ejecuto para comprobar que sigue mostrando los outputs correctamente.